# VectorBT Integration — NQ ORB Strategy

Implements the same ORB + EMA-20 + RSI-14 strategy using VectorBT's vectorized engine and compares results against the loop-based backtester (`strategy.py`).

**Goal:** Same performance metrics, lower computational overhead.

### Key differences vs. loop backtester
| Aspect | Loop (`strategy.py`) | VectorBT |
|---|---|---|
| Signal computation | Per-bar Python loop | Vectorized pandas |
| Stop checking | Bar high/low | OHLC (same) |
| Same-bar SL+TP | Always takes SL | Open-price ordering |
| Entry timing | Next bar's open | Next bar's open (shifted signal) |
| Force exit | 15:45 close | 15:45 close signal |

In [1]:
import time
import warnings
import numpy as np
import pandas as pd
import vectorbt as vbt
warnings.filterwarnings('ignore')

from data_fetch import fetch_data
from strategy import (
    run_backtest, _ema, _rsi,
    OR_BARS, EMA_PERIOD, RSI_PERIOD,
    RSI_LONG_MIN, RSI_SHORT_MAX,
    ENTRY_CUTOFF_H, ENTRY_CUTOFF_M,
    FORCE_EXIT_H, FORCE_EXIT_M,
    SL_POINTS, TP_POINTS,
    MULTIPLIER, INITIAL_EQUITY,
)

CONTRACTS = 4  # floor(500 / (60 * 2)) = 4

df = fetch_data(refresh=False)
print(f"{len(df):,} bars  |  {df.index[0].date()} -> {df.index[-1].date()}")

[data_fetch] Loading cached data from data\NQ_5m.csv ...


[data_fetch] Loaded 222,295 rows (2014-12-19 -> 2026-03-17)
222,295 bars  |  2014-12-19 -> 2026-03-17


## 1. Loop Backtester (baseline)

In [2]:
t0 = time.perf_counter()
loop_trades, loop_equity = run_backtest(df)
loop_time = time.perf_counter() - t0

loop_n     = len(loop_trades)
loop_wins  = int((loop_trades['pnl'] > 0).sum())
loop_wr    = loop_wins / loop_n * 100
loop_pnl   = loop_trades['pnl'].sum()

print(f"Trades      : {loop_n:,}")
print(f"Win rate    : {loop_wr:.1f}%")
print(f"Net P&L     : ${loop_pnl:+,.2f}")
print(f"Final equity: ${loop_equity.iloc[-1]:,.2f}")
print(f"Run time    : {loop_time:.3f}s")

Trades      : 2,740
Win rate    : 44.7%
Net P&L     : $+74,944.00
Final equity: $124,944.00
Run time    : 286.703s


## 2. VectorBT — Signal Pre-computation

All signal logic is vectorized with pandas `groupby` / `transform`.

**Entry timing trick:** The original signal fires at bar `i` (close), entry is at bar `i+1` open.  
We shift the signal forward by 1 bar and use `price=df['open']` so VectorBT executes at the correct bar's open.

In [3]:
def build_signals(df):
    d = df.copy()
    d['ema20'] = _ema(d['close'], EMA_PERIOD)
    d['rsi']   = _rsi(d['close'], RSI_PERIOD)

    # Bar index within each session (0 = first bar)
    d['bar_idx'] = d.groupby(d.index.date).cumcount()

    # Opening range high/low: max/min of first OR_BARS bars, broadcast to whole day
    d['or_high'] = d.groupby(d.index.date)['high'].transform(
        lambda x: x.iloc[:OR_BARS].max()
    )
    d['or_low'] = d.groupby(d.index.date)['low'].transform(
        lambda x: x.iloc[:OR_BARS].min()
    )

    is_after_or = d['bar_idx'] >= OR_BARS
    is_before_cutoff = ~(
        (d.index.hour > ENTRY_CUTOFF_H) |
        ((d.index.hour == ENTRY_CUTOFF_H) & (d.index.minute >= ENTRY_CUTOFF_M))
    )

    long_raw = (
        is_after_or & is_before_cutoff &
        (d['close'] > d['or_high']) &
        (d['close'] > d['ema20'])   &
        (d['rsi']   > RSI_LONG_MIN)
    )
    short_raw = (
        is_after_or & is_before_cutoff &
        (d['close'] < d['or_low'])  &
        (d['close'] < d['ema20'])   &
        (d['rsi']   < RSI_SHORT_MAX)
    )

    # Keep only the first signal per day (long OR short)
    any_signal     = long_raw | short_raw
    cumsum_per_day = any_signal.groupby(d.index.date).cumsum()
    is_first       = any_signal & (cumsum_per_day == 1)

    long_signal  = long_raw  & is_first
    short_signal = short_raw & is_first

    # Force-exit signal: close position at 15:45 regardless
    force_exit = (
        (d.index.hour   == FORCE_EXIT_H) &
        (d.index.minute == FORCE_EXIT_M)
    )

    # Holiday / early-close guard: only enter on days that have a 15:45 bar.
    # On early-close sessions the loop silently drops any position still open at
    # day end (no 15:45 bar = force exit never fires). VectorBT would instead
    # carry the position overnight. Filtering to full sessions prevents that.
    # Known limitation: on 9 early-close days the loop DID record a trade (SL/TP
    # hit before session end); VectorBT skips all early-close entries so those 9
    # trades are absent. This is irreconcilable without per-trade lookahead.
    dates     = pd.Series(d.index.date, index=d.index)
    days_with_force_exit = set(d.index[force_exit].date)
    full_session = dates.isin(days_with_force_exit)

    long_signal  = long_signal  & full_session
    short_signal = short_signal & full_session

    # Shift 1 bar: signal fires at bar i -> VectorBT executes at bar i+1 open
    long_exec  = long_signal.shift(1).fillna(False).astype(bool)
    short_exec = short_signal.shift(1).fillna(False).astype(bool)

    # Cross-day spill guard: entry bar must be on the same calendar date as the
    # original signal bar.
    same_day  = dates == dates.shift(1)
    long_exec  = long_exec  & same_day
    short_exec = short_exec & same_day

    # SL/TP as fraction of entry bar's open price
    sl_frac = SL_POINTS / d['open']   # ~0.003 for NQ ~20000
    tp_frac = TP_POINTS / d['open']   # ~0.006

    return d, long_exec, short_exec, sl_frac, tp_frac, force_exit


d, long_exec, short_exec, sl_frac, tp_frac, force_exit = build_signals(df)

print(f"Long entry signals : {long_exec.sum():,}")
print(f"Short entry signals: {short_exec.sum():,}")
print(f"Force-exit bars    : {force_exit.sum():,}")

Long entry signals : 1,489
Short entry signals: 1,242
Force-exit bars    : 2,797


## 3. VectorBT Portfolio

In [4]:
t0 = time.perf_counter()

# VectorBT treats instruments as stocks (full notional required per unit).
# NQ futures at $20,000 × 8 units = $160,000 per trade — far above $50k init_cash.
# Fix: use a large dummy cash balance so the constraint never binds.
# P&L is read from trade records directly, which is independent of init_cash.
DUMMY_CASH = 10_000_000

pf = vbt.Portfolio.from_signals(
    open=d['open'],
    high=d['high'],
    low=d['low'],
    close=d['close'],
    entries=long_exec,
    short_entries=short_exec,
    exits=force_exit,
    short_exits=force_exit,
    sl_stop=sl_frac,
    tp_stop=tp_frac,
    price=d['open'],              # execute at current bar's open (= next bar of signal)
    size=CONTRACTS * MULTIPLIER,  # 4 contracts x $2/pt -> PnL in dollars
    size_type='amount',
    init_cash=DUMMY_CASH,
    fees=0.0,
    freq='5min',
)

vbt_time = time.perf_counter() - t0

vbt_records = pf.trades.records_readable
vbt_n    = len(vbt_records)
vbt_wins = int((vbt_records['PnL'] > 0).sum())
vbt_wr   = vbt_wins / vbt_n * 100 if vbt_n else 0
vbt_pnl  = vbt_records['PnL'].sum()

print(f"Trades      : {vbt_n:,}")
print(f"Win rate    : {vbt_wr:.1f}%")
print(f"Net P&L     : ${vbt_pnl:+,.2f}")
print(f"Final equity: ${INITIAL_EQUITY + vbt_pnl:,.2f}")
print(f"Run time    : {vbt_time:.3f}s")

Trades      : 2,731
Win rate    : 45.1%
Net P&L     : $+67,447.51
Final equity: $117,447.51
Run time    : 10.669s


## 4. Side-by-Side Comparison

In [5]:
speedup = loop_time / vbt_time

comparison = pd.DataFrame({
    'Loop': [
        loop_n, loop_wins,
        f"{loop_wr:.1f}%", f"${loop_pnl:+,.0f}",
        f"${loop_equity.iloc[-1]:,.0f}", f"{loop_time:.3f}s", '1.0x',
    ],
    'VectorBT': [
        vbt_n, vbt_wins,
        f"{vbt_wr:.1f}%", f"${vbt_pnl:+,.0f}",
        f"${INITIAL_EQUITY + vbt_pnl:,.0f}", f"{vbt_time:.3f}s", f"{speedup:.1f}x",
    ],
    'Delta': [
        vbt_n - loop_n, vbt_wins - loop_wins,
        f"{vbt_wr - loop_wr:+.1f}pp", f"${vbt_pnl - loop_pnl:+,.0f}",
        f"${(INITIAL_EQUITY + vbt_pnl) - loop_equity.iloc[-1]:+,.0f}", '', '',
    ],
}, index=['Trades', 'Wins', 'Win rate', 'Net P&L', 'Final equity', 'Run time', 'Speedup'])

print(comparison.to_string())
print(f"\nKnown residual differences (irreconcilable):")
print(f"  -9 trades : loop records 9 early-close-day trades where SL/TP hit before")
print(f"              session end; VectorBT skips all early-close entries.")
print(f"  -$7.5k PnL: above 9 trades + same-bar SL/TP resolution (loop always takes")
print(f"              SL; VectorBT may take TP — requires custom Numba to fix).")

                  Loop  VectorBT    Delta
Trades            2740      2731       -9
Wins              1225      1232        7
Win rate         44.7%     45.1%   +0.4pp
Net P&L       $+74,944  $+67,448  $-7,496
Final equity  $124,944  $117,448  $-7,496
Run time      286.703s   10.669s         
Speedup           1.0x     26.9x         

Known residual differences (irreconcilable):
  -9 trades : loop records 9 early-close-day trades where SL/TP hit before
              session end; VectorBT skips all early-close entries.
  -$7.5k PnL: above 9 trades + same-bar SL/TP resolution (loop always takes
              SL; VectorBT may take TP — requires custom Numba to fix).


## 5. VectorBT Stats

In [6]:
pf.stats()

Start                         2014-12-19 09:30:00-05:00
End                           2026-03-17 12:25:00-04:00
Period                                771 days 20:35:00
Start Value                                  10000000.0
End Value                               10067447.507754
Total Return [%]                               0.674475
Benchmark Return [%]                          482.23383
Max Gross Exposure [%]                         2.082852
Total Fees Paid                                     0.0
Max Drawdown [%]                               0.193001
Max Drawdown Duration                 223 days 07:20:00
Total Trades                                       2731
Total Closed Trades                                2731
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                  45.111681
Best Trade [%]                                 2.918027
Worst Trade [%]                               -2

## 6. Equity Curve Overlay

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# VectorBT daily equity
vbt_eq = pf.value().resample('B').last().ffill()

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')

ax.plot(loop_equity.index, loop_equity.values,
        color='#00d4aa', linewidth=1.5, label='Loop backtester')
ax.plot(vbt_eq.index, vbt_eq.values,
        color='#ff9500', linewidth=1.0, linestyle='--', alpha=0.85, label='VectorBT')
ax.axhline(INITIAL_EQUITY, color='#555', linewidth=0.8, linestyle=':')

ax.set_title('Equity Curve: Loop vs VectorBT', color='white', fontsize=13)
ax.set_ylabel('Equity ($)', color='white')
ax.legend(fontsize=9, facecolor='#1a1a2e', labelcolor='white')
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/05_vbt_vs_loop.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> output/05_vbt_vs_loop.png')

Saved -> output/05_vbt_vs_loop.png
